# Photographic Paper Label Generator

**Purpose:** Develop a standalone Streamlit app that reads a CSV of sample data, generates diamond glyphs and radar charts, and produces formatted PDF label sheets for photographic paper samples.

**Output:** A `app.py` (Streamlit app) that runs independently of Jupyter/VS Code.

---
## Notebook Organization
- **Part 1** — Source file, outputs, and dependencies
- **Part 2** — Glyph rendering (radar charts + diamond glyphs)
- **Part 3** — Page formatting (8.5×11", 6mm margins, 3×4 grid of 6.5×6.5 cm squares)
- **Part 4** — Label content formatting
- **Part 5** — Sort order
- **Part 6** — Key cell (key.png, bottom-right of every page)
- **Part 7** — Error handling and UI
- **Part 8** — Export: write final `app.py`

---
## Part 1 — Source File, Outputs, and Dependencies

In [1]:
# Part 1: Install and import all dependencies
# This cell verifies the environment and documents required packages.
# The standalone app (app.py) will be deployable via: pip install -r requirements.txt

%pip install streamlit pandas numpy matplotlib reportlab Pillow -q

import os
import io
import math
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Polygon
from PIL import Image
from reportlab.lib.pagesizes import letter
from reportlab.lib.units import cm, mm
from reportlab.pdfgen import canvas as rl_canvas
from reportlab.lib import colors

print('✅ All dependencies imported successfully')
print()
print('Required CSV fields:')
REQUIRED_FIELDS = [
    'CatalogNumber', 'Manufacturer', 'Brand', 'Year',
    'Surface', 'TextureDescription', 'GlossDescription',
    'ColorDescription', 'ThicknessDescription',
    'Roughness', 'GlossUnits', 'WarmthAtDmin', 'Thickness_mm',
    'Fluorescence', 'DateIsApproximate', 'IsResinCoated',
    'HasProcessingInstructions', 'Backprint',
]
for f in REQUIRED_FIELDS:
    print(f'  - {f}')

# Normalization ranges (fixed, from reference code)
NORM_RANGES = {
    'bstar_base': {'lower': -6.13, 'upper': 31.45},   # WarmthAtDmin -> top axis
    'gloss':      {'lower': 0.19,  'upper': 123.55},  # GlossUnits -> right axis (inverted)
    'roughness':  {'lower': 0.005, 'upper': 0.373},   # Roughness -> bottom axis
    'thickness':  {'lower': 0.011, 'upper': 0.458},   # Thickness_mm -> left axis
}

# Glyph colors
GLYPH_FILL       = '#AED6F1'  # light blue
GLYPH_BORDER_NORMAL = '#1A5276'  # dark blue
GLYPH_BORDER_HIGH_FLUOR = '#0096FF'  # bright blue (Fluorescence > 1.75)
FLUOR_THRESHOLD  = 1.75

# Page / grid constants
PAGE_W_IN  = 8.5
PAGE_H_IN  = 11.0
MARGIN_MM  = 6.0
COLS       = 3
ROWS       = 4
SQUARES_PER_PAGE = COLS * ROWS          # 12 total; bottom-right is always the key
LABELS_PER_PAGE  = SQUARES_PER_PAGE - 1  # 11 data labels per page
SQUARE_CM  = 6.5

print()
print(f'Page: {PAGE_W_IN}" x {PAGE_H_IN}"  |  Margin: {MARGIN_MM}mm  |  Grid: {COLS}x{ROWS}  |  Square: {SQUARE_CM}cm x {SQUARE_CM}cm')
print(f'Labels per page: {LABELS_PER_PAGE}  (bottom-right square reserved for key on every page)')


[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
✅ All dependencies imported successfully

Required CSV fields:
  - CatalogNumber
  - Manufacturer
  - Brand
  - Year
  - Surface
  - TextureDescription
  - GlossDescription
  - ColorDescription
  - ThicknessDescription
  - Roughness
  - GlossUnits
  - WarmthAtDmin
  - Thickness_mm
  - Fluorescence
  - DateIsApproximate
  - IsResinCoated
  - HasProcessingInstructions
  - Backprint

Page: 8.5" x 11.0"  |  Margin: 6.0mm  |  Grid: 3x4  |  Square: 6.5cm x 6.5cm
Labels per page: 11  (bottom-right square reserved for key on every page)


---
## Part 2 — Glyph Rendering

Generates both **Radar Charts** (with gridlines) and **Diamond Glyphs** (plain, for label use).

Axis order: Top = WarmthAtDmin, Right = GlossUnits (inverted), Bottom = Roughness, Left = Thickness_mm.

Files saved to `{working_dir}/Radar Charts/{CatalogNumber}.png` and `{working_dir}/Diamond Glyphs/{CatalogNumber}.png`.

In [2]:
# Part 2: Glyph rendering functions

def normalize(value, lower, upper, invert=False):
    """Normalize a value to [0.1, 1.0] within [lower, upper]. Optionally invert."""
    try:
        v = float(value)
    except (TypeError, ValueError):
        return 0.1
    if np.isnan(v):
        return 0.1
    normed = (v - lower) / (upper - lower)
    if invert:
        normed = 1.0 - normed
    return float(np.clip(normed, 0.1, 1.0))


def compute_glyph_values(row):
    """Return (top, right, bottom, left) normalized values for one CSV row."""
    top    = normalize(row['WarmthAtDmin'], NORM_RANGES['bstar_base']['lower'],
                       NORM_RANGES['bstar_base']['upper'], invert=False)
    right  = normalize(row['GlossUnits'],   NORM_RANGES['gloss']['lower'],
                       NORM_RANGES['gloss']['upper'],     invert=True)   # inverted!
    bottom = normalize(row['Roughness'],    NORM_RANGES['roughness']['lower'],
                       NORM_RANGES['roughness']['upper'], invert=False)
    left   = normalize(row['Thickness_mm'], NORM_RANGES['thickness']['lower'],
                       NORM_RANGES['thickness']['upper'], invert=False)
    return top, right, bottom, left


def render_glyph(top, right, bottom, left,
                 border_color=GLYPH_BORDER_NORMAL,
                 fill_color=GLYPH_FILL,
                 size_px=300,
                 radar=False):
    """
    Render a glyph to a PNG bytes buffer.

    The glyph canvas is SQUARE_CM x SQUARE_CM with center at the axis intersection.
    Axis extent = SQUARE_CM / 2 on each side; a value of 1.0 reaches the border.

    radar=True adds gridlines and axis lines (for Radar Charts folder).
    radar=False is the clean diamond (for Diamond Glyphs folder and labels).
    """
    dpi = 150
    fig_size = size_px / dpi
    fig, ax = plt.subplots(figsize=(fig_size, fig_size), dpi=dpi)
    fig.patch.set_alpha(0.0)
    ax.set_facecolor('none')

    cx, cy = 0.5, 0.5
    sf = 0.45  # scale so 1.0 just touches border

    pts = [
        (cx,          cy + top    * sf),  # top
        (cx + right  * sf, cy         ),  # right
        (cx,          cy - bottom * sf),  # bottom
        (cx - left   * sf, cy         ),  # left
    ]

    if radar:
        for level in [0.2, 0.4, 0.6, 0.8, 1.0]:
            gpts = [
                (cx,           cy + level * sf),
                (cx + level * sf, cy          ),
                (cx,           cy - level * sf),
                (cx - level * sf, cy          ),
            ]
            ax.add_patch(Polygon(gpts, closed=True, facecolor='none',
                                 edgecolor='#888888', linewidth=0.5, alpha=0.4, zorder=1))
        for ep in [(cx, cy + sf), (cx + sf, cy), (cx, cy - sf), (cx - sf, cy)]:
            ax.plot([cx, ep[0]], [cy, ep[1]], color='#888888', linewidth=0.8, alpha=0.5, zorder=2)

    polygon = Polygon(pts, closed=True,
                      facecolor=fill_color,
                      edgecolor=border_color,
                      linewidth=2.5, alpha=1.0, zorder=5)
    ax.add_patch(polygon)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.axis('off')
    plt.tight_layout(pad=0)

    buf = io.BytesIO()
    plt.savefig(buf, format='png', transparent=True, dpi=dpi, bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    buf.seek(0)
    return buf


def generate_glyphs(df, working_dir, progress_callback=None):
    """
    Generate radar + diamond PNGs for all rows in df.
    Returns df with added columns: 'radar_path', 'diamond_path',
    'glyph_top', 'glyph_right', 'glyph_bottom', 'glyph_left'.
    """
    radar_dir   = os.path.join(working_dir, 'Radar Charts')
    diamond_dir = os.path.join(working_dir, 'Diamond Glyphs')
    os.makedirs(radar_dir,   exist_ok=True)
    os.makedirs(diamond_dir, exist_ok=True)

    radar_paths   = []
    diamond_paths = []
    tops, rights, bottoms, lefts = [], [], [], []

    for i, (_, row) in enumerate(df.iterrows()):
        cat = str(row['CatalogNumber']).replace('/', '_').replace('\\', '_')
        fluor = pd.to_numeric(row.get('Fluorescence', np.nan), errors='coerce')
        border = GLYPH_BORDER_HIGH_FLUOR if (not np.isnan(fluor) and fluor > FLUOR_THRESHOLD) else GLYPH_BORDER_NORMAL

        top, right, bottom, left = compute_glyph_values(row)
        tops.append(top); rights.append(right); bottoms.append(bottom); lefts.append(left)

        # Radar
        r_path = os.path.join(radar_dir, f'{cat}.png')
        buf = render_glyph(top, right, bottom, left, border_color=border, fill_color=GLYPH_FILL, radar=True)
        with open(r_path, 'wb') as f:
            f.write(buf.read())
        radar_paths.append(r_path)

        # Diamond
        d_path = os.path.join(diamond_dir, f'{cat}.png')
        buf = render_glyph(top, right, bottom, left, border_color=border, fill_color=GLYPH_FILL, radar=False)
        with open(d_path, 'wb') as f:
            f.write(buf.read())
        diamond_paths.append(d_path)

        if progress_callback:
            progress_callback(i + 1, len(df))

    df = df.copy()
    df['radar_path']   = radar_paths
    df['diamond_path'] = diamond_paths
    df['glyph_top']    = tops
    df['glyph_right']  = rights
    df['glyph_bottom'] = bottoms
    df['glyph_left']   = lefts
    return df


print('✅ Glyph rendering functions defined')
print('  Axis order: Top=WarmthAtDmin, Right=GlossUnits(inverted), Bottom=Roughness, Left=Thickness_mm')
print(f'  Fluorescence threshold for bright border: {FLUOR_THRESHOLD}')

✅ Glyph rendering functions defined
  Axis order: Top=WarmthAtDmin, Right=GlossUnits(inverted), Bottom=Roughness, Left=Thickness_mm
  Fluorescence threshold for bright border: 1.75


---
## Part 3 — Page Formatting

- Paper: 8.5 × 11" (letter)
- Margins: 6mm on all sides
- Grid: 3 columns × 4 rows = 12 squares of 6.5 × 6.5 cm each
- Year printed bold, 14pt, left-aligned below grid
- Page number right-aligned below grid, 10.5pt
- Font: Aptos (fallback: Arial)

In [3]:
# Part 3: Page geometry helpers

from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.colors import HexColor

# ---------- Font setup ----------
_FONT_REGISTERED = False

def _register_fonts():
    global _FONT_REGISTERED, FONT_NAME, FONT_BOLD_NAME
    if _FONT_REGISTERED:
        return
    # Try Aptos first (Windows 11 / Office 2021+)
    aptos_candidates = [
        (r'C:\Windows\Fonts\Aptos.ttf',     r'C:\Windows\Fonts\AptosBd.ttf'),
        (r'C:\Windows\Fonts\aptos.ttf',     r'C:\Windows\Fonts\aptosBd.ttf'),
    ]
    for reg, bold in aptos_candidates:
        if os.path.exists(reg) and os.path.exists(bold):
            pdfmetrics.registerFont(TTFont('Aptos', reg))
            pdfmetrics.registerFont(TTFont('Aptos-Bold', bold))
            FONT_NAME      = 'Aptos'
            FONT_BOLD_NAME = 'Aptos-Bold'
            _FONT_REGISTERED = True
            print('✅ Font: Aptos registered')
            return
    # Fallback: use ReportLab built-in Helvetica (maps Arial visually)
    FONT_NAME      = 'Helvetica'
    FONT_BOLD_NAME = 'Helvetica-Bold'
    _FONT_REGISTERED = True
    print('ℹ️  Aptos not found — using Helvetica (Arial equivalent)')


_register_fonts()

# ---------- Geometry ----------
PAGE_W = PAGE_W_IN * 72   # points
PAGE_H = PAGE_H_IN * 72
MARGIN = MARGIN_MM * mm   # reportlab mm -> points
SQUARE = SQUARE_CM * cm   # 6.5cm in points

# Grid origin: top-left corner of the grid, in ReportLab coords (y=0 at bottom)
GRID_LEFT   = MARGIN
GRID_TOP    = PAGE_H - MARGIN            # y of grid top edge
GRID_RIGHT  = GRID_LEFT + COLS * SQUARE
GRID_BOTTOM = GRID_TOP  - ROWS * SQUARE

def square_origin(col, row):
    """
    Return (x, y) of the bottom-left corner of the square at grid position
    (col, row) where col in [0,2] and row in [0,3], row=0 is TOP row.
    """
    x = GRID_LEFT + col * SQUARE
    y = GRID_TOP  - (row + 1) * SQUARE
    return x, y


def draw_grid(c):
    """Draw the 3×4 grid border lines on a ReportLab canvas."""
    c.setStrokeColorRGB(0, 0, 0)
    c.setLineWidth(0.5)
    # Horizontal lines
    for row in range(ROWS + 1):
        y = GRID_TOP - row * SQUARE
        c.line(GRID_LEFT, y, GRID_RIGHT, y)
    # Vertical lines
    for col in range(COLS + 1):
        x = GRID_LEFT + col * SQUARE
        c.line(x, GRID_BOTTOM, x, GRID_TOP)


def draw_page_footer(c, year_str, page_num, total_pages):
    """Draw year (bottom-left) and page number (bottom-right) below the grid."""
    footer_y = GRID_BOTTOM - 4 * mm  # just below grid

    # Year — bold, 14pt, left-aligned to grid left
    c.setFont(FONT_BOLD_NAME, 14)
    c.setFillColorRGB(0, 0, 0)
    c.drawString(GRID_LEFT, footer_y, str(year_str))

    # Page number — 10.5pt, right-aligned to grid right
    c.setFont(FONT_NAME, 10.5)
    page_text = f'{page_num} of {total_pages}'
    c.drawRightString(GRID_RIGHT, footer_y, page_text)


print(f'✅ Page geometry computed')
print(f'   Grid left={GRID_LEFT:.1f}pt  top={GRID_TOP:.1f}pt  right={GRID_RIGHT:.1f}pt  bottom={GRID_BOTTOM:.1f}pt')
print(f'   Square size: {SQUARE:.2f}pt ({SQUARE_CM}cm)')

ℹ️  Aptos not found — using Helvetica (Arial equivalent)
✅ Page geometry computed
   Grid left=17.0pt  top=775.0pt  right=569.8pt  bottom=38.0pt
   Square size: 184.25pt (6.5cm)


---
## Part 4 — Label Formatting

Each label square contains:
- Diamond glyph centered in the square, axes to borders
- Text overlay (left-aligned) with tags per field
- `PM# CatalogNumber` bold, right-aligned, pinned to bottom-right corner

In [4]:
# Part 4: Label content rendering

from reportlab.platypus import Paragraph
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_RIGHT

TEXT_MARGIN = 2 * mm   # inner padding from square border
TEXT_FONT_SIZE = 6.5   # pt — tight fit for 6.5cm square
PM_FONT_SIZE   = 7.5
PM_MARGIN      = 1.5 * mm  # from bottom/right edge of square


def _safe(val, fallback='NaN'):
    """Return string value, replacing blank/nan with fallback."""
    if val is None:
        return fallback
    s = str(val).strip()
    if s == '' or s.lower() == 'nan':
        return fallback
    return s


def _safe_text(val):
    """Return string value; return empty string if blank (for text fields)."""
    if val is None:
        return ''
    s = str(val).strip()
    if s.lower() == 'nan':
        return ''
    return s


def build_surface_line(row):
    """
    Build the Surface description string, skipping blank fields and
    avoiding consecutive commas/dashes.
    """
    parts = [
        _safe_text(row.get('Surface', '')),
        _safe_text(row.get('TextureDescription', '')),
        _safe_text(row.get('GlossDescription', '')),
        _safe_text(row.get('ColorDescription', '')),
        _safe_text(row.get('ThicknessDescription', '')),
    ]
    non_empty = [p for p in parts if p]
    if not non_empty:
        return 'Surface: '
    # First part joined with ' - ', rest with ', '
    first = non_empty[0]
    rest  = ', '.join(non_empty[1:])
    if rest:
        return f'Surface: {first} - {rest}'
    return f'Surface: {first}'


def build_label_lines(row):
    """
    Return a list of (text, bold) tuples for drawing on a label.
    bold=True triggers bold font for that line.
    """
    lines = []
    # Line 1: Manufacturer, Brand (bold)
    mfr  = _safe_text(row.get('Manufacturer', ''))
    brand = _safe_text(row.get('Brand', ''))
    header = ', '.join(p for p in [mfr, brand] if p)
    lines.append((header, True))

    # Line 2: Surface description (wrappable)
    lines.append((build_surface_line(row), False))

    # Lines 3-7: numeric fields
    lines.append((f'Texture: {_safe(row.get("Roughness"))} Sq', False))
    lines.append((f'Reflectance: {_safe(row.get("GlossUnits"))} GU', False))
    lines.append((f'Base color: {_safe(row.get("WarmthAtDmin"))} b*', False))
    lines.append((f'Thickness: {_safe(row.get("Thickness_mm"))} mm', False))
    lines.append((f'Fluor: {_safe(row.get("Fluorescence"))} AUC', False))

    # Lines 8-11: categorical fields
    lines.append((f'Year is apx: {_safe_text(row.get("DateIsApproximate", "")) or "NaN"}', False))
    lines.append((f'RC: {_safe_text(row.get("IsResinCoated", "")) or "NaN"}', False))
    lines.append((f'Instructions: {_safe_text(row.get("HasProcessingInstructions", "")) or "NaN"}', False))
    lines.append((f'Backprint: {_safe_text(row.get("Backprint", "")) or "NaN"}', False))

    return lines


def draw_glyph_on_label(c, x, y, diamond_path):
    """
    Place the diamond glyph PNG centered in the square.
    Glyph is drawn so axes span to the square edges (crop if needed).
    x, y = bottom-left of square in points.
    """
    if not diamond_path or not os.path.exists(diamond_path):
        return
    # The glyph PNG was rendered at full square size. Draw it filling the square.
    c.drawImage(diamond_path, x, y, width=SQUARE, height=SQUARE,
                mask='auto', preserveAspectRatio=True, anchor='c')


def draw_label_text(c, x, y, row, leading=8.5):
    """
    Draw label text lines inside the square at (x,y) bottom-left.
    """
    lines = build_label_lines(row)
    text_x = x + TEXT_MARGIN
    text_y = y + SQUARE - TEXT_MARGIN  # start near top

    for text, bold in lines:
        font = FONT_BOLD_NAME if bold else FONT_NAME
        size = TEXT_FONT_SIZE

        # Wrap text to fit square width
        max_width = SQUARE - 2 * TEXT_MARGIN
        wrapped = _wrap_text(c, text, font, size, max_width)

        for line in wrapped:
            text_y -= leading
            if text_y < y + PM_FONT_SIZE + 2 * PM_MARGIN:
                break  # don't overlap PM# line
            c.setFont(font, size)
            c.setFillColorRGB(0, 0, 0)
            c.drawString(text_x, text_y, line)

    # PM# — bold, right-aligned, fixed at bottom-right
    cat = _safe_text(row.get('CatalogNumber', ''))
    pm_text = f'PM# {cat}'
    pm_x = x + SQUARE - PM_MARGIN
    pm_y = y + PM_MARGIN
    c.setFont(FONT_BOLD_NAME, PM_FONT_SIZE)
    c.setFillColorRGB(0, 0, 0)
    c.drawRightString(pm_x, pm_y, pm_text)


def _wrap_text(c, text, font, size, max_width):
    """Simple word-wrap: split text into lines that fit within max_width."""
    words = text.split()
    lines = []
    current = ''
    for word in words:
        candidate = (current + ' ' + word).strip()
        if c.stringWidth(candidate, font, size) <= max_width:
            current = candidate
        else:
            if current:
                lines.append(current)
            current = word
    if current:
        lines.append(current)
    return lines if lines else ['']


def draw_blank_square(c, x, y):
    """Draw an empty square (just border, no content)."""
    # The grid lines handle borders; this is a no-op for now.
    pass


print('✅ Label formatting functions defined')

✅ Label formatting functions defined


---
## Part 5 — Sort Order

Labels are sorted by: **Year → Manufacturer → Brand → Surface → Texture (Roughness)**.

Pages break on Year. Labels for the same Year flow left-to-right, top-to-bottom across rows.
Remaining squares on a Year-page are blank (borders only). The key appears in the bottom-right of every page.

In [5]:
# Part 5: Sort and page-break logic

SORT_COLUMNS = ['Year', 'Manufacturer', 'Brand', 'Surface', 'Roughness']


def sort_dataframe(df):
    """Sort df per the label specifications."""
    sort_cols = [c for c in SORT_COLUMNS if c in df.columns]
    if not sort_cols:
        return df
    # Convert Year to numeric for correct sort order
    df = df.copy()
    if 'Year' in df.columns:
        df['_year_sort'] = pd.to_numeric(df['Year'], errors='coerce')
        sort_cols = ['_year_sort'] + [c for c in sort_cols if c != 'Year']
    df = df.sort_values(sort_cols, na_position='last')
    if '_year_sort' in df.columns:
        df = df.drop(columns=['_year_sort'])
    return df.reset_index(drop=True)


def build_pages(df):
    """
    Split sorted df into pages.
    Each page has a year and up to LABELS_PER_PAGE label rows.
    Returns list of {'year': str, 'rows': [row_dict or None]}.
    None marks blank squares.
    """
    pages = []
    if df.empty:
        return pages

    # Group by Year
    df['_year_key'] = df['Year'].astype(str)
    for year_key, group in df.groupby('_year_key', sort=False):
        records = group.drop(columns=['_year_key']).to_dict('records')
        # Split into chunks of LABELS_PER_PAGE
        for i in range(0, max(1, len(records)), LABELS_PER_PAGE):
            chunk = records[i:i + LABELS_PER_PAGE]
            # Pad with None to fill the page
            chunk += [None] * (LABELS_PER_PAGE - len(chunk))
            pages.append({'year': year_key, 'rows': chunk})

    df.drop(columns=['_year_key'], inplace=True, errors='ignore')
    return pages


print('✅ Sort and page-break functions defined')
print(f'   Sort order: {SORT_COLUMNS}')
print(f'   Labels per page: {LABELS_PER_PAGE}  (11 data + 1 key = 12 squares)')

✅ Sort and page-break functions defined
   Sort order: ['Year', 'Manufacturer', 'Brand', 'Surface', 'Roughness']
   Labels per page: 11  (11 data + 1 key = 12 squares)


---
## Part 6 — Key Cell

The bottom-right square of every page contains `key.png` from the working directory, centered within the square.

In [6]:
# Part 6: Key cell

KEY_FILENAME = 'key.png'

# Key position: bottom-right square = col 2, row 3
KEY_COL, KEY_ROW = COLS - 1, ROWS - 1


def draw_key_cell(c, working_dir):
    """
    Place key.png centered in the bottom-right square.
    If key.png is not found, the square is left blank.
    """
    key_path = os.path.join(working_dir, KEY_FILENAME)
    x, y = square_origin(KEY_COL, KEY_ROW)

    if not os.path.exists(key_path):
        # Leave blank (grid line already drawn)
        return

    # Center the image inside the square with a small inner margin
    inner_margin = 3 * mm
    img_size = SQUARE - 2 * inner_margin
    img_x = x + inner_margin
    img_y = y + inner_margin
    c.drawImage(key_path, img_x, img_y, width=img_size, height=img_size,
                mask='auto', preserveAspectRatio=True, anchor='c')


print('✅ Key cell function defined')
print(f'   Key image: {KEY_FILENAME}  |  Position: col={KEY_COL}, row={KEY_ROW} (bottom-right)')

✅ Key cell function defined
   Key image: key.png  |  Position: col=2, row=3 (bottom-right)


---
## Part 7 — PDF Assembly

Combines all parts: draws each page with grid, glyphs, text, footer, and key.

In [7]:
# Part 7: PDF assembly

def build_pdf(df_sorted, working_dir, output_path, progress_callback=None):
    """
    Build the full label PDF.
    df_sorted must already have 'diamond_path' column from generate_glyphs().
    """
    pages = build_pages(df_sorted)
    total_pages = len(pages)

    c = rl_canvas.Canvas(output_path, pagesize=(PAGE_W, PAGE_H))

    for page_idx, page in enumerate(pages):
        year_str = page['year']
        rows     = page['rows']  # list of 11 (row_dict or None)

        # Draw grid
        draw_grid(c)

        # Fill squares left-to-right, top-to-bottom (skip last = key)
        slot = 0  # index into rows[]
        for row_i in range(ROWS):
            for col_i in range(COLS):
                is_key_cell = (col_i == KEY_COL and row_i == KEY_ROW)
                if is_key_cell:
                    draw_key_cell(c, working_dir)
                    continue

                if slot >= len(rows):
                    slot += 1
                    continue

                row_data = rows[slot]
                slot += 1

                if row_data is None:
                    draw_blank_square(c, *square_origin(col_i, row_i))
                else:
                    sq_x, sq_y = square_origin(col_i, row_i)
                    # Clip to square before drawing glyph (handles oversized glyphs)
                    c.saveState()
                    p = c.beginPath()
                    p.rect(sq_x, sq_y, SQUARE, SQUARE)
                    c.clipPath(p, stroke=0, fill=0)
                    draw_glyph_on_label(c, sq_x, sq_y, row_data.get('diamond_path'))
                    c.restoreState()

                    draw_label_text(c, sq_x, sq_y, row_data)

        # Footer
        draw_page_footer(c, year_str, page_idx + 1, total_pages)

        if progress_callback:
            progress_callback(page_idx + 1, total_pages)

        if page_idx < total_pages - 1:
            c.showPage()

    c.save()
    print(f'✅ PDF saved: {output_path}  ({total_pages} pages)')
    return output_path


print('✅ PDF assembly function defined')

✅ PDF assembly function defined


---
## Part 8 — Error Handling and Validation

In [8]:
# Part 8: Validation helpers (used by both notebook and the Streamlit app)

def validate_csv(df):
    """
    Validate the loaded dataframe.
    Returns (ok: bool, warnings: list[str], errors: list[str]).
    """
    errors   = []
    warnings = []

    # Check required columns exist
    missing_cols = [f for f in REQUIRED_FIELDS if f not in df.columns]
    if missing_cols:
        errors.append(f'Missing required columns: {missing_cols}')
        return False, warnings, errors

    # Check Manufacturer and Brand for blanks
    for field in ['Manufacturer', 'Brand']:
        blank_mask = df[field].isna() | (df[field].astype(str).str.strip() == '') | \
                     (df[field].astype(str).str.lower() == 'nan')
        if blank_mask.any():
            bad_cats = df.loc[blank_mask, 'CatalogNumber'].tolist()
            warnings.append(
                f'Blank {field} in CatalogNumber(s): {", ".join(str(c) for c in bad_cats)}'
            )

    ok = len(errors) == 0
    return ok, warnings, errors


def check_prerequisites(working_dir, csv_path, output_dir):
    """Return list of missing prerequisite messages."""
    missing = []
    if not working_dir or not os.path.isdir(working_dir):
        missing.append('Working directory not set or does not exist.')
    if not csv_path:
        missing.append('No CSV file selected.')
    if not output_dir:
        missing.append('No output folder selected.')
    return missing


print('✅ Validation functions defined')

✅ Validation functions defined


---
## Part 9 — Export: Write `app.py` (Streamlit Standalone App)

Running this cell writes the complete standalone Streamlit app to `label_generator/app.py`.
Users can run it with: `streamlit run app.py`

In [9]:
APP_CODE = '''
"""Label Generator — Streamlit App

Usage:
    streamlit run app.py

Requires:
    pip install -r requirements.txt
"""
import io
import os
import math
import datetime
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from reportlab.lib.pagesizes import letter
from reportlab.lib.units import cm, mm
from reportlab.pdfgen import canvas as rl_canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
import streamlit as st

# ============================================================
# CONSTANTS
# ============================================================
REQUIRED_FIELDS = [
    "CatalogNumber", "Manufacturer", "Brand", "Year",
    "Surface", "TextureDescription", "GlossDescription",
    "ColorDescription", "ThicknessDescription",
    "Roughness", "GlossUnits", "WarmthAtDmin", "Thickness_mm",
    "Fluorescence", "DateIsApproximate", "IsResinCoated",
    "HasProcessingInstructions", "Backprint",
]

NORM_RANGES = {
    "bstar_base": {"lower": -6.13,  "upper": 31.45},
    "gloss":      {"lower": 0.19,   "upper": 123.55},
    "roughness":  {"lower": 0.005,  "upper": 0.373},
    "thickness":  {"lower": 0.011,  "upper": 0.458},
}

GLYPH_FILL             = "#AED6F1"
GLYPH_BORDER_NORMAL    = "#1A5276"
GLYPH_BORDER_HIGH_FLUOR = "#0096FF"
FLUOR_THRESHOLD        = 1.75

PAGE_W_IN  = 8.5
PAGE_H_IN  = 11.0
MARGIN_MM  = 6.0
COLS       = 3
ROWS       = 4
SQUARE_CM  = 6.5
LABELS_PER_PAGE = COLS * ROWS - 1  # 11
KEY_COL, KEY_ROW = COLS - 1, ROWS - 1
KEY_FILENAME = "key.png"
SORT_COLUMNS = ["Year", "Manufacturer", "Brand", "Surface", "Roughness"]
TEXT_FONT_SIZE = 6.5
PM_FONT_SIZE   = 7.5
TEXT_MARGIN = 2 * mm
PM_MARGIN   = 1.5 * mm

PAGE_W = PAGE_W_IN * 72
PAGE_H = PAGE_H_IN * 72
MARGIN = MARGIN_MM * mm
SQUARE = SQUARE_CM * cm
GRID_LEFT   = MARGIN
GRID_TOP    = PAGE_H - MARGIN
GRID_RIGHT  = GRID_LEFT + COLS * SQUARE
GRID_BOTTOM = GRID_TOP  - ROWS * SQUARE

# ============================================================
# FONT SETUP
# ============================================================
_fonts_registered = False
FONT_NAME = "Helvetica"
FONT_BOLD_NAME = "Helvetica-Bold"

def _register_fonts():
    global _fonts_registered, FONT_NAME, FONT_BOLD_NAME
    if _fonts_registered:
        return
    candidates = [
        (r"C:\\Windows\\Fonts\\Aptos.ttf", r"C:\\Windows\\Fonts\\AptosBd.ttf"),
        (r"C:\\Windows\\Fonts\\aptos.ttf", r"C:\\Windows\\Fonts\\aptosBd.ttf"),
    ]
    for reg, bold in candidates:
        if os.path.exists(reg) and os.path.exists(bold):
            pdfmetrics.registerFont(TTFont("Aptos", reg))
            pdfmetrics.registerFont(TTFont("Aptos-Bold", bold))
            FONT_NAME = "Aptos"
            FONT_BOLD_NAME = "Aptos-Bold"
            break
    _fonts_registered = True

# ============================================================
# NORMALIZATION
# ============================================================
def _normalize(value, lower, upper, invert=False):
    try:
        v = float(value)
    except (TypeError, ValueError):
        return 0.1
    if np.isnan(v):
        return 0.1
    n = (v - lower) / (upper - lower)
    if invert:
        n = 1.0 - n
    return float(np.clip(n, 0.1, 1.0))

def _compute_glyph_values(row):
    top    = _normalize(row["WarmthAtDmin"], NORM_RANGES["bstar_base"]["lower"], NORM_RANGES["bstar_base"]["upper"])
    right  = _normalize(row["GlossUnits"],   NORM_RANGES["gloss"]["lower"],      NORM_RANGES["gloss"]["upper"],      invert=True)
    bottom = _normalize(row["Roughness"],    NORM_RANGES["roughness"]["lower"],  NORM_RANGES["roughness"]["upper"])
    left   = _normalize(row["Thickness_mm"], NORM_RANGES["thickness"]["lower"],  NORM_RANGES["thickness"]["upper"])
    return top, right, bottom, left

# ============================================================
# GLYPH RENDERING
# ============================================================
def _render_glyph_png(top, right, bottom, left, border_color, fill_color, radar=False):
    dpi = 150
    fig_size = 300 / dpi
    fig, ax = plt.subplots(figsize=(fig_size, fig_size), dpi=dpi)
    fig.patch.set_alpha(0.0)
    ax.set_facecolor("none")
    cx, cy, sf = 0.5, 0.5, 0.45
    pts = [
        (cx,         cy + top    * sf),
        (cx + right  * sf, cy        ),
        (cx,         cy - bottom * sf),
        (cx - left   * sf, cy        ),
    ]
    if radar:
        for lv in [0.2, 0.4, 0.6, 0.8, 1.0]:
            gpts = [(cx, cy + lv*sf), (cx+lv*sf, cy), (cx, cy-lv*sf), (cx-lv*sf, cy)]
            ax.add_patch(Polygon(gpts, closed=True, facecolor="none", edgecolor="#888888",
                                 linewidth=0.5, alpha=0.4, zorder=1))
        for ep in [(cx, cy+sf), (cx+sf, cy), (cx, cy-sf), (cx-sf, cy)]:
            ax.plot([cx, ep[0]], [cy, ep[1]], color="#888888", linewidth=0.8, alpha=0.5, zorder=2)
    ax.add_patch(Polygon(pts, closed=True, facecolor=fill_color, edgecolor=border_color,
                         linewidth=2.5, alpha=1.0, zorder=5))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xticks([]); ax.set_yticks([]); ax.axis("off")
    plt.tight_layout(pad=0)
    buf = io.BytesIO()
    plt.savefig(buf, format="png", transparent=True, dpi=dpi, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    buf.seek(0)
    return buf

def generate_glyphs(df, working_dir, st_progress=None):
    """Generate radar + diamond PNGs. Adds radar_path, diamond_path columns."""
    radar_dir   = os.path.join(working_dir, "Radar Charts")
    diamond_dir = os.path.join(working_dir, "Diamond Glyphs")
    os.makedirs(radar_dir,   exist_ok=True)
    os.makedirs(diamond_dir, exist_ok=True)
    r_paths, d_paths = [], []
    n = len(df)
    for i, (_, row) in enumerate(df.iterrows()):
        cat = str(row["CatalogNumber"]).replace("/", "_").replace("\\\\", "_")
        fluor = pd.to_numeric(row.get("Fluorescence", np.nan), errors="coerce")
        border = GLYPH_BORDER_HIGH_FLUOR if (not np.isnan(fluor) and fluor > FLUOR_THRESHOLD) else GLYPH_BORDER_NORMAL
        top, right, bottom, left = _compute_glyph_values(row)
        r_path = os.path.join(radar_dir,   f"{cat}.png")
        d_path = os.path.join(diamond_dir, f"{cat}.png")
        buf = _render_glyph_png(top, right, bottom, left, border, GLYPH_FILL, radar=True)
        with open(r_path, "wb") as f: f.write(buf.read())
        buf = _render_glyph_png(top, right, bottom, left, border, GLYPH_FILL, radar=False)
        with open(d_path, "wb") as f: f.write(buf.read())
        r_paths.append(r_path)
        d_paths.append(d_path)
        if st_progress is not None:
            st_progress.progress((i + 1) / n, text=f"Generating glyphs {i+1}/{n}...")
    df = df.copy()
    df["radar_path"]   = r_paths
    df["diamond_path"] = d_paths
    return df

# ============================================================
# VALIDATION
# ============================================================
def validate_csv(df):
    errors, warnings = [], []
    missing = [f for f in REQUIRED_FIELDS if f not in df.columns]
    if missing:
        errors.append(f"Missing required columns: {missing}")
        return False, warnings, errors
    for field in ["Manufacturer", "Brand"]:
        mask = df[field].isna() | (df[field].astype(str).str.strip() == "") | \
               (df[field].astype(str).str.lower() == "nan")
        if mask.any():
            bad = df.loc[mask, "CatalogNumber"].tolist()
            warnings.append(f"Blank {field} in CatalogNumber(s): {', '.join(str(c) for c in bad)}")
    return True, warnings, errors

# ============================================================
# SORT
# ============================================================
def sort_dataframe(df):
    sc = [c for c in SORT_COLUMNS if c in df.columns]
    if not sc:
        return df
    df = df.copy()
    if "Year" in df.columns:
        df["_year_sort"] = pd.to_numeric(df["Year"], errors="coerce")
        sc = ["_year_sort"] + [c for c in sc if c != "Year"]
    df = df.sort_values(sc, na_position="last")
    df.drop(columns=["_year_sort"], inplace=True, errors="ignore")
    return df.reset_index(drop=True)

def build_pages(df):
    pages = []
    df["_year_key"] = df["Year"].astype(str)
    for year_key, group in df.groupby("_year_key", sort=False):
        records = group.drop(columns=["_year_key"]).to_dict("records")
        for i in range(0, max(1, len(records)), LABELS_PER_PAGE):
            chunk = records[i:i + LABELS_PER_PAGE]
            chunk += [None] * (LABELS_PER_PAGE - len(chunk))
            pages.append({"year": year_key, "rows": chunk})
    df.drop(columns=["_year_key"], inplace=True, errors="ignore")
    return pages

# ============================================================
# LABEL TEXT
# ============================================================
def _safe(val, fallback="NaN"):
    if val is None: return fallback
    s = str(val).strip()
    return fallback if s == "" or s.lower() == "nan" else s

def _safe_text(val):
    if val is None: return ""
    s = str(val).strip()
    return "" if s.lower() == "nan" else s

def _build_surface_line(row):
    parts = [_safe_text(row.get(k, "")) for k in
             ["Surface", "TextureDescription", "GlossDescription", "ColorDescription", "ThicknessDescription"]]
    ne = [p for p in parts if p]
    if not ne: return "Surface: "
    return f"Surface: {ne[0]}" + (f" - {', '.join(ne[1:])}" if len(ne) > 1 else "")

def _build_label_lines(row):
    mfr   = _safe_text(row.get("Manufacturer", ""))
    brand = _safe_text(row.get("Brand", ""))
    header = ", ".join(p for p in [mfr, brand] if p)
    return [
        (header, True),
        (_build_surface_line(row), False),
        (f"Texture: {_safe(row.get('Roughness'))} Sq", False),
        (f"Reflectance: {_safe(row.get('GlossUnits'))} GU", False),
        (f"Base color: {_safe(row.get('WarmthAtDmin'))} b*", False),
        (f"Thickness: {_safe(row.get('Thickness_mm'))} mm", False),
        (f"Fluor: {_safe(row.get('Fluorescence'))} AUC", False),
        (f"Year is apx: {_safe_text(row.get('DateIsApproximate', '')) or 'NaN'}", False),
        (f"RC: {_safe_text(row.get('IsResinCoated', '')) or 'NaN'}", False),
        (f"Instructions: {_safe_text(row.get('HasProcessingInstructions', '')) or 'NaN'}", False),
        (f"Backprint: {_safe_text(row.get('Backprint', '')) or 'NaN'}", False),
    ]

def _wrap_text_rl(c, text, font, size, max_width):
    words = text.split()
    lines, current = [], ""
    for word in words:
        candidate = (current + " " + word).strip()
        if c.stringWidth(candidate, font, size) <= max_width:
            current = candidate
        else:
            if current: lines.append(current)
            current = word
    if current: lines.append(current)
    return lines if lines else [""]

# ============================================================
# PAGE DRAWING
# ============================================================
def _square_origin(col, row):
    return GRID_LEFT + col * SQUARE, GRID_TOP - (row + 1) * SQUARE

def _draw_grid(c):
    c.setStrokeColorRGB(0, 0, 0)
    c.setLineWidth(0.5)
    for r in range(ROWS + 1):
        y = GRID_TOP - r * SQUARE
        c.line(GRID_LEFT, y, GRID_RIGHT, y)
    for col in range(COLS + 1):
        x = GRID_LEFT + col * SQUARE
        c.line(x, GRID_BOTTOM, x, GRID_TOP)

def _draw_page_footer(c, year_str, page_num, total_pages):
    footer_y = GRID_BOTTOM - 4 * mm
    c.setFont(FONT_BOLD_NAME, 14)
    c.setFillColorRGB(0, 0, 0)
    c.drawString(GRID_LEFT, footer_y, str(year_str))
    c.setFont(FONT_NAME, 10.5)
    c.drawRightString(GRID_RIGHT, footer_y, f"{page_num} of {total_pages}")

def _draw_glyph(c, x, y, diamond_path):
    if not diamond_path or not os.path.exists(diamond_path): return
    c.drawImage(diamond_path, x, y, width=SQUARE, height=SQUARE,
                mask="auto", preserveAspectRatio=True, anchor="c")

def _draw_label_text(c, x, y, row, leading=8.5):
    lines = _build_label_lines(row)
    tx = x + TEXT_MARGIN
    ty = y + SQUARE - TEXT_MARGIN
    for text, bold in lines:
        font = FONT_BOLD_NAME if bold else FONT_NAME
        wrapped = _wrap_text_rl(c, text, font, TEXT_FONT_SIZE, SQUARE - 2*TEXT_MARGIN)
        for line in wrapped:
            ty -= leading
            if ty < y + PM_FONT_SIZE + 2*PM_MARGIN: break
            c.setFont(font, TEXT_FONT_SIZE)
            c.setFillColorRGB(0, 0, 0)
            c.drawString(tx, ty, line)
    cat = _safe_text(row.get("CatalogNumber", ""))
    c.setFont(FONT_BOLD_NAME, PM_FONT_SIZE)
    c.setFillColorRGB(0, 0, 0)
    c.drawRightString(x + SQUARE - PM_MARGIN, y + PM_MARGIN, f"PM# {cat}")

def _draw_key_cell(c, working_dir):
    key_path = os.path.join(working_dir, KEY_FILENAME)
    x, y = _square_origin(KEY_COL, KEY_ROW)
    if not os.path.exists(key_path): return
    inner = 3 * mm
    c.drawImage(key_path, x+inner, y+inner, width=SQUARE-2*inner, height=SQUARE-2*inner,
                mask="auto", preserveAspectRatio=True, anchor="c")

def build_pdf(df_sorted, working_dir, output_path, st_progress=None):
    _register_fonts()
    pages = build_pages(df_sorted)
    total = len(pages)
    c = rl_canvas.Canvas(output_path, pagesize=(PAGE_W, PAGE_H))
    for pi, page in enumerate(pages):
        _draw_grid(c)
        slot = 0
        for ri in range(ROWS):
            for ci in range(COLS):
                if ci == KEY_COL and ri == KEY_ROW:
                    _draw_key_cell(c, working_dir)
                    continue
                if slot >= len(page["rows"]): slot += 1; continue
                row_data = page["rows"][slot]; slot += 1
                if row_data is None: continue
                sx, sy = _square_origin(ci, ri)
                c.saveState()
                p = c.beginPath()
                p.rect(sx, sy, SQUARE, SQUARE)
                c.clipPath(p, stroke=0, fill=0)
                _draw_glyph(c, sx, sy, row_data.get("diamond_path"))
                c.restoreState()
                _draw_label_text(c, sx, sy, row_data)
        _draw_page_footer(c, page["year"], pi+1, total)
        if st_progress is not None:
            st_progress.progress((pi+1)/total, text=f"Building PDF page {pi+1}/{total}...")
        if pi < total - 1:
            c.showPage()
    c.save()
    return output_path

# ============================================================
# STREAMLIT UI
# ============================================================
st.set_page_config(page_title="Label Generator", layout="wide")
st.title("📄 Photographic Paper Label Generator")

with st.sidebar:
    st.header("⚙️ Settings")
    csv_file      = st.file_uploader("Upload CSV file", type=["csv"])
    working_dir   = st.text_input("Working directory (for glyph output + key.png)", value="")
    output_dir    = st.text_input("Output folder (for PDF)", value="")

# ---- Prerequisite check ----
prereq_missing = []
if not csv_file:
    prereq_missing.append("No CSV file uploaded.")
if not working_dir or not os.path.isdir(working_dir):
    prereq_missing.append("Working directory is missing or does not exist.")
if not output_dir:
    prereq_missing.append("No output folder specified.")
elif not os.path.isdir(output_dir):
    try:
        os.makedirs(output_dir, exist_ok=True)
    except Exception:
        prereq_missing.append(f"Cannot create output folder: {output_dir}")

if prereq_missing:
    st.warning("Please complete the following before processing:")
    for m in prereq_missing:
        st.error(m)

process_btn = st.button("🚀 Process Labels", disabled=bool(prereq_missing))

if csv_file and not prereq_missing:
    # Load and validate CSV
    try:
        for enc in ["utf-8", "latin-1", "cp1252"]:
            try:
                df_raw = pd.read_csv(csv_file, encoding=enc)
                break
            except UnicodeDecodeError:
                csv_file.seek(0)
        ok, warns, errs = validate_csv(df_raw)
        if errs:
            for e in errs:
                st.error(e)
            st.stop()
        if warns:
            st.warning("Data warnings found:")
            for w in warns:
                st.warning(w)
            proceed = st.radio(
                "Some required fields have blank data. Proceed anyway?",
                ["Yes — continue", "No — fix CSV first"],
                index=1
            )
            if proceed == "No — fix CSV first":
                st.stop()
        st.success(f"✅ CSV loaded: {len(df_raw)} records")
    except Exception as ex:
        st.error(f"Could not load CSV: {ex}")
        st.stop()

if process_btn:
    progress = st.progress(0, text="Starting...")
    status   = st.empty()

    status.info("Sorting records...")
    df_sorted = sort_dataframe(df_raw)

    status.info("Generating glyphs...")
    df_sorted = generate_glyphs(df_sorted, working_dir, st_progress=progress)

    status.info("Building PDF...")
    today = datetime.datetime.now().strftime("%m%d%Y")
    pdf_name = f"Labels_{today}.pdf"
    pdf_path = os.path.join(output_dir, pdf_name)
    build_pdf(df_sorted, working_dir, pdf_path, st_progress=progress)

    progress.empty()
    status.success(f"✅ Done! PDF saved to: {pdf_path}")

    with open(pdf_path, "rb") as f:
        st.download_button("⬇️ Download PDF", data=f, file_name=pdf_name, mime="application/pdf")
'''

out_path = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                        'label_generator', 'app.py')
# Write from the notebook's own directory
nb_dir = os.path.abspath('')  # current working directory in notebook
app_path = os.path.join(nb_dir, 'app.py')

with open(app_path, 'w', encoding='utf-8') as f:
    f.write(APP_CODE.strip())

print(f'✅ app.py written to: {app_path}')
print()
print('To run the standalone app:')
print('  cd label_generator')
print('  pip install -r requirements.txt')
print('  streamlit run app.py')

✅ app.py written to: c:\Users\pm\code\label_generator\app.py

To run the standalone app:
  cd label_generator
  pip install -r requirements.txt
  streamlit run app.py


---
## Quick Test (Optional)

Run this cell to do a quick smoke test with synthetic data to verify glyph and PDF generation work before connecting a real CSV.

In [10]:
# Quick smoke test with synthetic data
import tempfile

_test_dir = tempfile.mkdtemp(prefix='label_test_')
print(f'Test output dir: {_test_dir}')

_test_data = {
    'CatalogNumber':           ['PM2230', 'PM2231', 'PM2232'],
    'Manufacturer':            ['Kodak',  'Kodak',  'Kodak'],
    'Brand':                   ['Velox',  'Velox',  'Velox'],
    'Year':                    [1928,     1928,     1929],
    'Surface':                 ['F',      'F',      'F'],
    'TextureDescription':      ['smooth', 'smooth', 'smooth'],
    'GlossDescription':        ['glossy', 'glossy', 'glossy'],
    'ColorDescription':        ['white',  'white',  'white'],
    'ThicknessDescription':    ['single weight', 'single weight', 'single weight'],
    'Roughness':               [0.102,   0.102,   0.102],
    'GlossUnits':              [12.05,   12.05,   12.05],
    'WarmthAtDmin':            [7.05,    7.05,    7.05],
    'Thickness_mm':            [0.050,   0.050,   0.050],
    'Fluorescence':            [0.045,   2.0,     0.045],  # PM2231 triggers bright border
    'DateIsApproximate':       ['Yes',   'Yes',   'Yes'],
    'IsResinCoated':           ['No',    'No',    'No'],
    'HasProcessingInstructions': ['Yes', 'Yes',   'Yes'],
    'Backprint':               ['No',    'No',    'No'],
}

_df = pd.DataFrame(_test_data)
ok, warns, errs = validate_csv(_df)
print(f'Validation: ok={ok}, warnings={warns}, errors={errs}')

_df_sorted = sort_dataframe(_df)
_df_sorted = generate_glyphs(_df_sorted, _test_dir)

print('Generated glyph paths:')
for _, r in _df_sorted.iterrows():
    print(f"  Radar:   {r['radar_path']}  exists={os.path.exists(r['radar_path'])}")
    print(f"  Diamond: {r['diamond_path']}  exists={os.path.exists(r['diamond_path'])}")

_pdf_path = os.path.join(_test_dir, 'Labels_TEST.pdf')
build_pdf(_df_sorted, _test_dir, _pdf_path)
print(f'PDF size: {os.path.getsize(_pdf_path):,} bytes')
print(f'✅ Smoke test passed — open {_pdf_path} to inspect')

Test output dir: C:\Users\pm\AppData\Local\Temp\label_test_5skive57
Validation: ok=True, warnings=[], errors=[]
Generated glyph paths:
  Radar:   C:\Users\pm\AppData\Local\Temp\label_test_5skive57\Radar Charts\PM2230.png  exists=True
  Diamond: C:\Users\pm\AppData\Local\Temp\label_test_5skive57\Diamond Glyphs\PM2230.png  exists=True
  Radar:   C:\Users\pm\AppData\Local\Temp\label_test_5skive57\Radar Charts\PM2231.png  exists=True
  Diamond: C:\Users\pm\AppData\Local\Temp\label_test_5skive57\Diamond Glyphs\PM2231.png  exists=True
  Radar:   C:\Users\pm\AppData\Local\Temp\label_test_5skive57\Radar Charts\PM2232.png  exists=True
  Diamond: C:\Users\pm\AppData\Local\Temp\label_test_5skive57\Diamond Glyphs\PM2232.png  exists=True
✅ PDF saved: C:\Users\pm\AppData\Local\Temp\label_test_5skive57\Labels_TEST.pdf  (2 pages)
PDF size: 12,243 bytes
✅ Smoke test passed — open C:\Users\pm\AppData\Local\Temp\label_test_5skive57\Labels_TEST.pdf to inspect
